# Clusterizer based on K-means

In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import MiniBatchKMeans
import joblib

## Dataset class

In [ ]:
class LatentSpaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.data = []

        self._load_data()

    def _load_data(self):
        for filename in os.listdir(self.root_dir):
            _, ext = os.path.splitext(filename)
            if ext == '.pt':
                ls_path = os.path.join(self.root_dir, filename)
                self.data.append(ls_path)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ls_path = self.data[idx]
        latent_space = torch.load(ls_path)
        
        if self.transform:
            latent_space = self.transform(latent_space)

        return latent_space

## Clusterizer class

In [ ]:
class Clusterizer:
    def __init__(self, n_clusters=8, max_iter=100, tol=0.0, patience=10):
        self.model = MiniBatchKMeans(
            n_clusters=n_clusters,
            max_iter=max_iter,
            tol=tol,
            max_no_improvement=patience
        )

    def fit(self, dataset, batch_size=32):
        dataloader = DataLoader(dataset, batch_size=batch_size)
        for batch in dataloader:
            self.model.partial_fit(batch)

    def predict(self, latent_spaces):
        return self.model.predict(latent_spaces)

    def save(self, filename):
        filename = os.path.join('out', f'{filename}.pkl')
        joblib.dump(self, filename, 3)

    @staticmethod
    def load(filename):
        return joblib.load(filename)